# 02: Efficiency Analysis

**Goal:** Benchmark transit agency efficiency using capital per vehicle, per mile, and per capita metrics.

**Data:** FTA NTD Capital Expenses, 2024 — 1,000 records

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('../data/ntd_capital_expenses.csv')
numeric_cols = ['guideway','stations','administrative_buildings','maintenance_buildings',
                'passenger_vehicles','other_vehicles','fare_collection_equipment',
                'communication_information','other','total','primary_uza_population','mode_voms']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print(f"Records: {len(df)}")
print(f"Total capital: ${df['total'].sum()/1e9:.1f}B")
df.head()

## Cost per Vehicle by Mode

In [ ]:
# Only rows with vehicles
df['total_vehicles'] = df['passenger_vehicles'] + df['other_vehicles']
vehicle_df = df[df['total_vehicles'] > 0].copy()
vehicle_df['cost_per_vehicle'] = vehicle_df['total'] / vehicle_df['total_vehicles']

mode_eff = vehicle_df.groupby('mode_name').agg({
    'cost_per_vehicle': ['mean','median','count'],
    'total_vehicles': 'sum'
}).round(0)
mode_eff.columns = ['_'.join(c) for c in mode_eff.columns]
mode_eff = mode_eff.sort_values('cost_per_vehicle_mean', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

mode_eff['cost_per_vehicle_mean'].plot(kind='barh', ax=axes[0], color='#e67e22')
axes[0].set_title('Mean Cost per Vehicle by Mode')
axes[0].set_xlabel('USD / Vehicle')

vehicle_df.boxplot(column='cost_per_vehicle', by='mode_name', ax=axes[1], rot=45)
axes[1].set_title('Cost per Vehicle Distribution by Mode')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

## Cost per Capita by Metro Area

In [ ]:
pop_df = df[df['primary_uza_population'] > 0].copy()
pop_df['cost_per_capita'] = pop_df['total'] / pop_df['primary_uza_population']

top_metro = pop_df.groupby('uza_name')['cost_per_capita'].sum().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 7))
top_metro.plot(kind='barh', ax=ax, color='#3498db')
ax.set_title('Capital per Capita by Metro Area (Top 15)')
ax.set_xlabel('USD / Person')
plt.tight_layout()
plt.show()

## Agency Benchmarking (Top 20 by Total Capital)

In [ ]:
top_agencies = df.groupby('agency')['total'].sum().nlargest(20).index
bench = df[df['agency'].isin(top_agencies)].copy()
bench['cost_per_vehicle'] = bench['total'] / (bench['passenger_vehicles'] + bench['other_vehicles']).replace(0, np.nan)

fig, ax = plt.subplots(figsize=(12, 10))
bench_pivot = bench.groupby('agency')['cost_per_vehicle'].mean().sort_values(ascending=False)
bench_pivot.plot(kind='barh', ax=ax, color='#2ecc71')
ax.set_title('Cost per Vehicle — Top 20 Agencies by Capital')
ax.set_xlabel('USD / Vehicle')
plt.tight_layout()
plt.show()